In [ ]:
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib as jl
from cebra import CEBRA
import cebra.datasets
import scipy.io
import os.path
import plotly
import plotly.io as pio
import copy
import torch
from sklearn.model_selection import train_test_split

pio.renderers.default = "notebook"  # or "jupyterlab"

In [ ]:
# our data
data_folder = r'/media/NAS1/Katja/LightLevel'
data_name = 'main_data'
curr_folder = r'/media/NAS1/Katja/LightLevel'
data_path = os.path.join(data_folder,data_name)
data = scipy.io.loadmat(data_path)
data.keys()

In [ ]:
# extract different data type and reshape as necessary
speed = data['ALL_SPEED']
behv = data['ALL_BEH'][:,:-1]

base_pattern = [1, 2, 3, 1, 2, 3]
species = np.repeat(base_pattern, 4)
species = np.repeat(species, 29)
base_pattern2 = [1, 1, 1, 0, 0, 0]
light = np.repeat(base_pattern2, 4)
light = np.repeat(light, 29)

speed_r = speed.reshape(-1, 450)
behv_r = behv.reshape(-1,1)
behv_r = behv_r.squeeze()
speed_p = speed_r[:,1:145]
speed_r = speed_r[:,150:]

behaviour = np.column_stack((species, light))
behaviour.shape

base_pattern3 = np.repeat([0, 1, 2, 3], 29)  # Each number repeated 29 times
stimulus = np.tile(base_pattern3, 6)

In [ ]:
# subset for specific stimulus or species only
inds = np.where(stimulus < 3)[0]
speed_r_sub = speed_r[inds,:]
speed_p_sub = speed_p[inds,:]
behv_r_sub = behv_r[inds]
behaviour_sub = behaviour[inds,:]
stimuli_sub = stimulus[inds]
light_sub = light[inds]
species_sub = species[inds]

In [ ]:
# Split data and labels (labels we use later!)

split_idx = int(0.8 * len(speed_r)) #suggest: 5%-20% depending on your dataset size

train_data = speed_r[:split_idx]
valid_data = speed_r[split_idx:]

train_continuous_label = behaviour[:split_idx]
valid_continuous_label = behaviour[split_idx:]

split_idx_sub = int(0.8 * len(speed_r_sub)) #suggest: 5%-20% depending on your dataset size

train_data_sub = speed_r_sub[:split_idx_sub]
valid_data_sub = speed_r_sub[split_idx_sub:]

train_continuous_label_sub = behaviour_sub[:split_idx_sub]
valid_continuous_label_sub = behaviour_sub[split_idx_sub:]

In [ ]:
class BehaviorData:
    def __init__(self, neural, continuous_index):
        self.neural = neural          # shape (696,)
        self.continuous_index = continuous_index  # shape (696, 3)
    
    def __len__(self):
        """Returns the number of samples (696)"""
        return len(self.neural)
    
    def __getitem__(self, idx):
        """Enable indexing like all_data[idx] or all_data[inds,:]"""
        # Handle both simple indices and tuple indices (like [inds,:])
        if isinstance(idx, tuple):
            # Case: [inds,:] - idx will be (inds, slice(None, None, None))
            return BehaviorData(
                neural=self.neural[idx[0]],
                continuous_index=self.continuous_index[idx[0]],  # or idx[0],: if needed
            )
        else:
            # Case: [inds] - simple indexing
            
            return BehaviorData(
                neural=self.neural[idx],
                continuous_index=self.continuous_index[idx],
            )
# Create your data structure
all_data = BehaviorData(speed_r, behaviour)
all_data_sub = all_data[inds,:]

In [ ]:
# ############################## grid search if results aren't satisfactory - only once!
os.makedirs('saved_models3')

# dataset1: full data set from light level paper
# dataset2: only black looming stimuli from light level paper

params_grid = dict(
    output_dimension = [3, 6],
    time_offsets = [5, 10],
    model_architecture='offset10-model',
    temperature_mode='constant',
    temperature=[0.1, 1.0],
    max_iterations=[5000],
    device='cuda_if_available',
    num_hidden_units = [32, 64],
    verbose = True)

# ++++++++++ CHANGE TO CORRECT DATASET AND SAME ++++++++++++++++
datasets = {"dataset3": train_data_sub}
# run the grid search
grid_search = cebra.grid_search.GridSearch()
grid_search.fit_models(datasets, params=params_grid, models_dir="saved_models3")

In [ ]:
#  Get the results +++++++ CHECK CORRECT DATASET NAME
cebra.allow_lazy_imports()
grid_search = cebra.grid_search.GridSearch()
df_results = grid_search.get_df_results(models_dir="saved_models2")

# Get the best model for a given dataset
best_model, best_model_name = grid_search.get_best_model(dataset_name="dataset2", models_dir="saved_models2")
print("The best model is:", best_model_name)

In [ ]:
# load the top model
save_path = "saved_models2"
model_name = f"{best_model_name}.pt"
model_path = os.path.join(curr_folder,save_path,model_name)
top_model = cebra.CEBRA.load(model_path)

#transform:
top_train_embedding = top_model.transform(train_data)
top_valid_embedding = top_model.transform(valid_data)

# plot the loss curve
ax = cebra.plot_loss(top_model)


# plot embeddings
fig = cebra.integrations.plotly.plot_embedding_interactive(top_train_embedding,
                                                           embedding_labels=train_continuous_label[:,0],
                                                           title = "top model - train",
                                                           markersize=3,
                                                           cmap = "rainbow")
fig.show()

fig = cebra.integrations.plotly.plot_embedding_interactive(top_valid_embedding,
                                                           embedding_labels=valid_continuous_label[:,0],
                                                           title = "top model - validation",
                                                           markersize=3,
                                                           cmap = "rainbow")
fig.show()


In [ ]:
top_model

In [ ]:
# fit models subset
cebra_time_model = CEBRA(model_architecture='offset10-model',
                        batch_size=512,
                        learning_rate=3e-4,
                        temperature=0.1,
                        output_dimension=6,
                        max_iterations=5000,
                        distance='cosine',
                        conditional='time',
                        device='cuda_if_available',
                        verbose=True,
                        time_offsets=5,
                        ) 
cebra_time_model.fit(speed_r_sub)
cebra_time_model.save("cebra_time_model_sub.pt")

neural_data = speed_r_sub  
behavior_data = np.zeros((speed_r_sub.shape[0], 2), dtype=np.float32)
behavior_data[:, 0] = behv_r_sub
neural_tensor = torch.FloatTensor(neural_data)
behavior_tensor = torch.FloatTensor(behavior_data)
cebra_behavior_model = CEBRA(model_architecture='offset10-model',
                        batch_size=512,
                        learning_rate=3e-4,
                        temperature=0.1,
                        output_dimension=6,
                        max_iterations=5000,
                        distance='cosine',
                        conditional='time_delta',
                        device='cuda_if_available',
                        verbose=True,
                        time_offsets=5,
                        ) 
cebra_behavior_model.fit(neural_tensor,behavior_tensor)
cebra_behavior_model.save("cebra_behavior_model_sub.pt")


cebra_hybrid_model = CEBRA(model_architecture='offset10-model',
                        batch_size=512,
                        learning_rate=3e-4,
                        temperature=0.1,
                        output_dimension=6,
                        max_iterations=5000,
                        distance='cosine',
                        conditional='time_delta',
                        device='cuda_if_available',
                        verbose=True,
                        time_offsets=5,
                        hybrid = True) 
cebra_hybrid_model.fit(neural_tensor,behavior_tensor)
cebra_hybrid_model.save("cebra_hybrid_model_sub.pt")

behv_shuffled_posdir_sub = np.random.permutation(behv_r_sub.squeeze())
neural_data = speed_r_sub  
behavior_data = np.zeros((behv_shuffled_posdir_sub.shape[0], 3), dtype=np.float32)
behavior_data[:, 0] = behv_shuffled_posdir_sub
neural_tensor = torch.FloatTensor(neural_data)
behavior_tensor = torch.FloatTensor(behavior_data)
cebra_behavior_shuffled_model = CEBRA(model_architecture='offset10-model',
                        batch_size=512,
                        learning_rate=3e-4,
                        temperature=0.1,
                        output_dimension=6,
                        max_iterations=5000,
                        distance='cosine',
                        conditional='time_delta',
                        device='cuda_if_available',
                        verbose=True,
                        time_offsets=5,
                        hybrid = True) 
cebra_behavior_shuffled_model.fit(neural_tensor,behavior_tensor)
cebra_behavior_shuffled_model.save("cebra_behavior_shuffled_model_sub.pt")

In [ ]:
# load models and get embeddings - SUBSET


# CEBRA-Time
# cebra_time_model = cebra.CEBRA.load("cebra_time_model.pt")
cebra_time_model = cebra.CEBRA.load("cebra_time_model_sub.pt", weights_only=False)
cebra_time = cebra_time_model.transform(speed_r_sub)

# CEBRA-Behavior
# cebra_behavior_model = cebra.CEBRA.load("cebra_behavior_model.pt")
cebra_behavior_model = cebra.CEBRA.load("cebra_behavior_model_sub.pt", weights_only=False)
cebra_behavior = cebra_behavior_model.transform(speed_r_sub)

# CEBRA-Hybrid
# cebra_hybrid_model = cebra.CEBRA.load("cebra_hybrid_model.pt")
cebra_hybrid_model = cebra.CEBRA.load("cebra_hybrid_model_sub.pt", weights_only=False)
cebra_hybrid = cebra_hybrid_model.transform(speed_r_sub)

# CEBRA-Behavior with shuffled labels
# cebra_behavior_shuffled_model = cebra.CEBRA.load("cebra_behavior_shuffled_model.pt")
cebra_behavior_shuffled_model = cebra.CEBRA.load("cebra_behavior_shuffled_model_sub.pt", weights_only=False)
cebra_behavior_shuffled = cebra_behavior_shuffled_model.transform(speed_r_sub)

In [ ]:
#look at specific conditions
behaviour = all_data_sub.continuous_index
bright = behaviour[:,1] == 1
dark = behaviour[:,1] == 0
# behavior type, species, light

fig = plt.figure(figsize=(10,2))

ax1 = plt.subplot(141, projection='3d')
ax2 = plt.subplot(142, projection='3d')
ax3 = plt.subplot(143, projection='3d')
ax4 = plt.subplot(144, projection='3d')

for dir, cmap in zip([bright, dark], ["Blues", "Oranges"]):
    ax1=cebra.plot_embedding(ax=ax1, embedding=cebra_behavior[dir,:], embedding_labels=behaviour[dir,0], markersize = 8, title='CEBRA-Behavior', cmap=cmap)
    ax2=cebra.plot_embedding(ax=ax2, embedding=cebra_behavior_shuffled[dir,:], embedding_labels=behaviour[dir,0],  markersize = 8,title='CEBRA-Shuffled', cmap=cmap)
    ax3=cebra.plot_embedding(ax=ax3, embedding=cebra_time[dir,:], embedding_labels=behaviour[dir,0],  markersize = 8,title='CEBRA-Time', cmap=cmap)
    ax4=cebra.plot_embedding(ax=ax4, embedding=cebra_hybrid[dir,:], embedding_labels=behaviour[dir,0], markersize = 8, title='CEBRA-Hybrid', cmap=cmap)

plt.show()

In [ ]:
#plot one specific embedding, check different dimensions to plot and different labels!!

fig = plt.figure(figsize=(12, 5))
ax = plt.subplot(121, projection = '3d')
# ax.set_title('x', fontsize=20, y=0)
ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.xaxis._axinfo["grid"]['color'] =  (1,1,1,0)
ax.yaxis._axinfo["grid"]['color'] =  (1,1,1,0)
ax.zaxis._axinfo["grid"]['color'] =  (1,1,1,0)
x = ax.scatter(cebra_hybrid[:,0],
               cebra_hybrid[:, 1],
               cebra_hybrid[:, 2],
               c=behaviour[:,0],
               cmap='rainbow',
               s=4,
               vmin=1,
               vmax=3)
xc = plt.colorbar(x, fraction=0.03, pad=0.1, ticks=np.linspace(0, 1, 1))

ax = plt.subplot(122, projection = '3d')
# ax.set_title('x', fontsize=20, y=0)
ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.xaxis._axinfo["grid"]['color'] =  (1,1,1,0)
ax.yaxis._axinfo["grid"]['color'] =  (1,1,1,0)
ax.zaxis._axinfo["grid"]['color'] =  (1,1,1,0)
y = ax.scatter(cebra_hybrid[:,1],
               cebra_hybrid[:, 3],
               cebra_hybrid[:, 5],
               c=behaviour[:,1],
               cmap='rainbow',
               s=4,
               vmin=0,
               vmax=1)
# ax.axis('off')
yc = plt.colorbar(y, fraction=0.03, pad=0.1, ticks=np.linspace(0, 1, 1))

# fig.savefig('cebra_behaviourshuffled_speciesLight.svg', format='svg', bbox_inches='tight')

In [ ]:
#cebra time plot dimensions 3,5,4

fig = plt.figure(figsize=(12, 5))
ax = plt.subplot(121, projection = '3d')
# ax.set_title('x', fontsize=20, y=0)
ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.xaxis._axinfo["grid"]['color'] =  (1,1,1,0)
ax.yaxis._axinfo["grid"]['color'] =  (1,1,1,0)
ax.zaxis._axinfo["grid"]['color'] =  (1,1,1,0)
x = ax.scatter(cebra_behavior_shuffled[:,3],
               cebra_behavior_shuffled[:, 4],
               cebra_behavior_shuffled[:, 5],
               c=stimuli_sub,
               cmap='rainbow',
               s=4,
               vmin=0,
               vmax=2)
xc = plt.colorbar(x, fraction=0.03, pad=0.1, ticks=np.linspace(0, 2, 1))

ax = plt.subplot(122, projection = '3d')
# ax.set_title('x', fontsize=20, y=0)
ax.xaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.yaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.zaxis.set_pane_color((1.0, 1.0, 1.0, 0.0))
ax.xaxis._axinfo["grid"]['color'] =  (1,1,1,0)
ax.yaxis._axinfo["grid"]['color'] =  (1,1,1,0)
ax.zaxis._axinfo["grid"]['color'] =  (1,1,1,0)
y = ax.scatter(cebra_behavior_shuffled[:,0],
               cebra_behavior_shuffled[:, 1],
               cebra_behavior_shuffled[:, 2],
               c=stimuli_sub,
               cmap='rainbow',
               s=4,
               vmin=0,
               vmax=2)
# ax.axis('off')
yc = plt.colorbar(y, fraction=0.03, pad=0.1, ticks=np.linspace(0, 2, 1))

# fig.savefig('cebra_behaviorshuffled_stimulus.svg', format='svg', bbox_inches='tight')

In [ ]:
# same thing but interactive
fig = cebra.integrations.plotly.plot_embedding_interactive(cebra_time,
                                                           embedding_labels=behaviour[:,1],
                                                           title = "top model - train",
                                                           markersize=3,
                                                           cmap = "rainbow")
fig.show()



Here new training - Hypothesis testing starts

In [ ]:
def split_data(data, test_ratio):
    split_idx = int(len(data) * (1 - test_ratio))
    neural_train = data.neural[:split_idx]
    neural_test = data.neural[split_idx:]
    label_train = data.continuous_index[:split_idx]
    label_test = data.continuous_index[split_idx:]
    
    # Remove .numpy() since the arrays are already NumPy
    return neural_train, neural_test, label_train, label_test

neural_train, neural_test, label_train, label_test = split_data(all_data_sub, 0.2)

In [ ]:
neural_data = speed_r_sub  
behavior_data = np.zeros((522, 3), dtype=np.float32)
behavior_data = behaviour
neural_tensor = torch.FloatTensor(neural_data)
behavior_tensor = torch.FloatTensor(behavior_data)

# Train CEBRA-Behavior models with both position and direction variables.
cebra_posdir_model = CEBRA(model_architecture='offset10-model',
                        batch_size=512,
                        learning_rate=3e-4,
                        temperature=0.1,
                        output_dimension=6,
                        max_iterations=5000,
                        distance='cosine',
                        conditional='time_delta',
                        device='cuda_if_available',
                        verbose=True,
                        time_offsets=5)
cebra_posdir_model.fit(neural_tensor, behavior_tensor.numpy())
cebra_posdir_model.save("cebra_posdir_model_sub.pt")

# Train CEBRA-Behavior models with position only.
cebra_pos_model = CEBRA(model_architecture='offset10-model',
                        batch_size=512,
                        learning_rate=3e-4,
                        temperature=0.1,
                        output_dimension=6,
                        max_iterations=5000,
                        distance='cosine',
                        device='cuda_if_available',
                        verbose=True,
                        time_offsets=5)
cebra_pos_model.fit(speed_r_sub, behaviour[:,0])
cebra_pos_model.save("cebra_pos_model_sub.pt")

# Train CEBRA-Behavior models with direction only.
cebra_dir_model = CEBRA(model_architecture='offset10-model',
                        batch_size=512,
                        learning_rate=3e-4,
                        temperature=0.1,
                        output_dimension=6,
                        max_iterations=5000,
                        distance='cosine',
                        device='cuda_if_available',
                        verbose=True,
                        time_offsets=5)
cebra_dir_model.fit(speed_r_sub, behaviour[:,1])
cebra_dir_model.save("cebra_dir_model_sub.pt")



In [ ]:
shuffled_pos = np.random.permutation(behaviour[:,0])
shuffled_dir = np.random.permutation(behaviour[:,1])
#shuffled_species = np.random.permutation(behaviour[:,2])

neural_data = speed_r_sub  
behavior_data = np.zeros((522, 3), dtype=np.float32)
behavior_data = np.random.permutation(behv_r_sub)
neural_tensor = torch.FloatTensor(neural_data)
behavior_tensor_sh = torch.FloatTensor(behavior_data)

# Train CEBRA-Behavior models with both position and direction variables.
cebra_posdir_model_sh = CEBRA(model_architecture='offset10-model',
                        batch_size=512,
                        learning_rate=3e-4,
                        temperature=0.1,
                        output_dimension=6,
                        max_iterations=5000,
                        distance='cosine',
                        conditional='time_delta',
                        device='cuda_if_available',
                        verbose=True,
                        time_offsets=5)
cebra_posdir_model_sh.fit(neural_tensor, behavior_tensor_sh.numpy())
cebra_posdir_model_sh.save("cebra_posdir_model_sub_sh.pt")

# Train CEBRA-Behavior models with position only.
cebra_pos_model_sh = CEBRA(model_architecture='offset10-model',
                        batch_size=512,
                        learning_rate=3e-4,
                        temperature=0.1,
                        output_dimension=6,
                        max_iterations=5000,
                        distance='cosine',
                        device='cuda_if_available',
                        verbose=True,
                        time_offsets=5)
cebra_pos_model_sh.fit(speed_r_sub, shuffled_pos)
cebra_pos_model_sh.save("cebra_pos_model_sub_sh.pt")

# Train CEBRA-Behavior models with direction only.
cebra_dir_model_sh = CEBRA(model_architecture='offset10-model',
                        batch_size=512,
                        learning_rate=3e-4,
                        temperature=0.1,
                        output_dimension=6,
                        max_iterations=5000,
                        distance='cosine',
                        device='cuda_if_available',
                        verbose=True,
                        time_offsets=5)
cebra_dir_model_sh.fit(speed_r_sub, shuffled_dir)
cebra_dir_model_sh.save("cebra_dir_model_sub_sh.pt")



In [ ]:
# We get train set embedding and test set embedding.
neural_data = speed_r_sub  
behavior_data = np.zeros((522, 3), dtype=np.float32)
behavior_data = np.random.permutation(behaviour)
neural_tensor = torch.FloatTensor(neural_data)
behavior_tensor_sh = torch.FloatTensor(behavior_data)

cebra_posdir_model = cebra.CEBRA.load("cebra_posdir_model_sub.pt", weights_only=False)
cebra_posdir = cebra_posdir_model.transform(neural_tensor)

cebra_pos_model = cebra.CEBRA.load("cebra_pos_model_sub.pt", weights_only=False)
cebra_pos = cebra_pos_model.transform(speed_r_sub)

cebra_dir_model = cebra.CEBRA.load("cebra_dir_model_sub.pt", weights_only=False)
cebra_dir = cebra_dir_model.transform(speed_r_sub)




cebra_posdir_model_sh = cebra.CEBRA.load("cebra_posdir_model_sub_sh.pt", weights_only=False)
cebra_posdir_sh = cebra_posdir_model_sh.transform(neural_tensor)

cebra_pos_model_sh = cebra.CEBRA.load("cebra_pos_model_sub_sh.pt", weights_only=False)
cebra_pos_sh = cebra_pos_model_sh.transform(speed_r_sub)

cebra_dir_model_sh = cebra.CEBRA.load("cebra_dir_model_sub_sh.pt", weights_only=False)
cebra_dir_sh = cebra_dir_model_sh.transform(speed_r_sub)


In [ ]:
behaviour2 = np.float32(behaviour) # depending on the format of the data used for training, one or the other works
mod = cebra_posdir_model_sh
behv = behaviour2
# behv = behaviour[:,1]
loss = cebra.sklearn.metrics.infonce_loss(mod, neural_tensor,behv,num_batches = 10)
gof = cebra.sklearn.metrics.infonce_to_goodness_of_fit(loss, model=mod)
print(loss,gof)

In [ ]:
cebra.sklearn.metrics.goodness_of_fit_history(cebra_dir_model)

In [ ]:
fig=plt.figure(figsize=(12,6))

ax1=plt.subplot(241, projection = '3d')
ax2=plt.subplot(242, projection = '3d')
ax3=plt.subplot(243, projection = '3d')
ax4=plt.subplot(244, projection = '3d')
ax5=plt.subplot(245, projection = '3d')
ax6=plt.subplot(246, projection = '3d')
ax7=plt.subplot(247, projection = '3d')
ax8=plt.subplot(248, projection = '3d')

ax1=cebra.plot_embedding(ax=ax1, embedding=cebra_pos, embedding_labels=behaviour[:,0], title='species only',markersize = 8,cmap = "rainbow")
ax2=cebra.plot_embedding(ax=ax2, embedding=cebra_dir, embedding_labels=behaviour[:,1], title='light level only',markersize = 8,cmap = "rainbow")
ax4=cebra.plot_embedding(ax=ax4, embedding=cebra_posdir, embedding_labels=behaviour[:,1], title='mixed',markersize = 8)
ax5=cebra.plot_embedding(ax=ax5, embedding=cebra_pos_sh, embedding_labels=behaviour[:,0], title='speciees, shuffled',markersize = 8,cmap = "rainbow")
ax6=cebra.plot_embedding(ax=ax6, embedding=cebra_dir_sh, embedding_labels=behaviour[:,1], title='light, shuffled',markersize = 8,cmap = "rainbow")
ax8=cebra.plot_embedding(ax=ax8, embedding=cebra_posdir_sh, embedding_labels=behaviour[:,1], title='mixed, shuffled',markersize = 8)

plt.show()

In [ ]:
#2D plots
fig = plt.figure(figsize=(12, 12))
ax = plt.subplot(221)
x = ax.scatter(cebra_posdir[:,5],
               cebra_posdir[:, 3],
               c=species_sub,
               cmap='rainbow',
               s=4,
               vmin=1,
               vmax=3)
xc = plt.colorbar(x, fraction=0.03, pad=0.1, ticks=np.linspace(0, 1, 1))

ax = plt.subplot(222)
x = ax.scatter(cebra_posdir[:,1],
               cebra_posdir[:, 2],
               c=light_sub,
               cmap='rainbow',
               s=4,
               vmin=0,
               vmax=1)

ax = plt.subplot(223)
x = ax.scatter(cebra_posdir[:,1],
               cebra_posdir[:, 5],
               c=species_sub,
               cmap='rainbow',
               s=4,
               vmin=1,
               vmax=3)
xc = plt.colorbar(x, fraction=0.03, pad=0.1, ticks=np.linspace(0, 1, 1))

ax = plt.subplot(224)
x = ax.scatter(cebra_posdir[:,1],
               cebra_posdir[:, 5],
               c=light_sub,
               cmap='rainbow',
               s=4,
               vmin=0,
               vmax=1)
# ax.axis('off')
# fig.savefig('cebra_mixedshuffled_speciesLight.svg', format='svg', bbox_inches='tight')

In [ ]:
fig = plt.figure(figsize=(5,5))
ax = plt.subplot(111)

ax = cebra.plot_loss(cebra_pos_model, color='deepskyblue', label='behaviour type', ax=ax)
ax = cebra.plot_loss(cebra_dir_model, color='deepskyblue', alpha=0.3, label='light', ax=ax)
ax = cebra.plot_loss(cebra_posdir_model, color='blue', alpha=0.6,label='mixed', ax=ax)

ax = cebra.plot_loss(cebra_pos_model_sh, color='gray', label='behaviour type, shuffled', ax=ax)
ax = cebra.plot_loss(cebra_dir_model_sh, color='gray', alpha=0.3, label='light, shuffled', ax=ax)
x = cebra.plot_loss(cebra_posdir_model_sh, color='red', alpha=0.6,label='mixed,shuffled', ax=ax)

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_xlabel('Iterations')
ax.set_ylabel('InfoNCE Loss')
plt.legend(bbox_to_anchor=(0.5,0.3), frameon = False)
plt.show()